# Domain 2: Tool Design & MCP Integration
## TS 2.4: Integrate MCP Servers into Claude Code and Agent Workflows
**Certification Focus**: CCAR-F Domain 2 (18% Exam Weight) | **Scenario Alignment**: Scenario 4 (Developer Productivity with Claude)

---

### 🎯 Key Architectural Principles
1. **MCP Resources vs Tools**:
   - **Resources**: Read-only, cacheable reference artifacts (e.g. database schema `catalog://schema/orders`, OpenAPI catalogs). Exposing these as Resources eliminates exploratory tool calls.
   - **Tools**: Stateful operations that perform actions or execute queries with dynamic parameters.
2. **Surgical File Editing**: Editing files via string replacement requires exact, unique anchor matches.
3. **Safe Fallback to Read + Write**: If an anchor match is non-unique (ambiguous) or fails, Claude must safely fall back to reading the full file and writing the full updated file.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 2.4 MCP Server Integration.")

### 📦 Step 2: Simulated MCP Resource Catalog & Safe Edit Function
We implement an MCP service exposing resources and a safe file editing function with fallback detection.

In [ ]:
# Step 2: MCP Service & Safe Edit Implementation
class DeveloperMCPService:
    """Simulates an MCP service offering read-only Resources and operational Tools."""
    RESOURCES = {
        "catalog://schema/orders": {
            "uri": "catalog://schema/orders",
            "name": "Orders Database Schema",
            "mimeType": "application/json",
            "text": json.dumps({
                "tables": {
                    "orders": ["id", "customer_id", "total_cents", "status", "created_at"],
                    "order_items": ["id", "order_id", "sku", "quantity", "unit_price_cents"],
                }
            }),
        },
        "catalog://docs/api_endpoints": {
            "uri": "catalog://docs/api_endpoints",
            "name": "API Route Catalog",
            "mimeType": "text/markdown",
            "text": "# Routes\n- GET /api/v1/orders/{id}\n- POST /api/v1/orders/{id}/refund",
        },
    }

    @classmethod
    def read_resource(cls, uri: str) -> Optional[Dict[str, Any]]:
        return cls.RESOURCES.get(uri)

    @classmethod
    def list_resources(cls) -> List[Dict[str, str]]:
        return [{"uri": v["uri"], "name": v["name"]} for v in cls.RESOURCES.values()]


def safe_edit_file(file_content: str, old_string: str, new_string: str) -> Tuple[bool, str]:
    """
    Safely edits file content. Detects non-unique anchors and triggers fallback to Read+Write.
    """
    match_count = file_content.count(old_string)
    if match_count == 0:
        return False, "Anchor string not found. Triggering Read + Write fallback."
    if match_count > 1:
        return False, f"Ambiguous edit: Anchor string found {match_count} times. Triggering Read + Write fallback."
    
    updated_content = file_content.replace(old_string, new_string, 1)
    return True, updated_content

print(" DeveloperMCPService and safe_edit_file ready.")

### 🧪 Step 3: Interactive Simulation
1. Read catalog schema resource (`catalog://schema/orders`).
2. Perform unique surgical edit -> Success.
3. Attempt ambiguous edit (multiple matches) -> Triggers fallback.

In [ ]:
# Step 3: Simulation
# 1. Read MCP Resource
schema_res = DeveloperMCPService.read_resource("catalog://schema/orders")
print("MCP Resource Read Result:", json.loads(schema_res["text"]))

# 2. Successful Unique Edit
code_sample = "def calculate_tax(amount):\n    rate = 0.05\n    return amount * rate\n"
ok, new_code = safe_edit_file(code_sample, "rate = 0.05", "rate = 0.08")
print("\nUnique Edit Status:", ok)
print("Updated Code:\n", new_code)

# 3. Ambiguous Edit (fails unique match)
ambiguous_code = "x = 1\nx = 1\n"
ok_amb, msg = safe_edit_file(ambiguous_code, "x = 1", "x = 2")
print("\nAmbiguous Edit Status:", ok_amb)
print("Fallback Message:", msg)

### 🔬 Step 4: Verification Assertions
We assert that:
1. MCP Resources can be read by URI without tool execution.
2. Unique strings are safely replaced.
3. Ambiguous anchors cleanly reject with a Read+Write fallback notice.

In [ ]:
# Step 4: Verification Checks
assert schema_res is not None
assert "orders" in schema_res["text"]

assert ok is True
assert "rate = 0.08" in new_code

assert ok_amb is False
assert "Triggering Read + Write fallback" in msg

print(" All verification assertions passed! MCP integration validated.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Expose static database schemas and documentation as **MCP Resources**, reserving Tools for active queries and actions.
- **Rule 2**: Surgical file edits require unique anchor matching.
- **Rule 3**: If an anchor match is ambiguous, always fall back to a full **Read + Write** workflow.